# Phase 2: Model Training

In this notebook, we will:
1. Load the preprocessed Alzheimer's dataset.
2. Fine-tune a pretrained EfficientNet-B4 model.
3. First, train only the classifier head.
4. Then, unfreeze the last 3 blocks and train with a lower learning rate.
5. Save the trained weights to `backend/models/efficientnet_adni.pth`.

## 1. Setup Configuration & Dataloaders

In [2]:
import os
import sys
import time
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from pathlib import Path

# Add the parent directory to sys.path to import backend modules
sys.path.append(os.path.abspath('..'))

from backend.core.preprocessing import get_dataloaders
from backend.core.model import get_model, freeze_base_model, unfreeze_last_blocks

ModuleNotFoundError: No module named 'backend'

In [ ]:
DATASET_DIR = "../dataset/Alzheimer_Dataset_V3/Unaugmented"
BATCH_SIZE = 32
NUM_WORKERS = 0 # Set to 0 on Windows to avoid multiprocessing issues in Jupyter
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_loader, val_loader, test_loader, class_to_idx = get_dataloaders(
    dataset_dir=DATASET_DIR, 
    batch_size=BATCH_SIZE, 
    num_workers=NUM_WORKERS
)

print(f"Classes: {class_to_idx}")

## 2. Define Training Logic

In [ ]:
def train_model(model, dataloaders, criterion, optimizer, num_epochs=10, device='cpu'):
    since = time.time()
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    best_acc = 0.0
    best_model_wts = model.state_dict()
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()
                
            running_loss = 0.0
            running_corrects = 0
            
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                        
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)
            
            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
                
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = model.state_dict()
                    
        print()
        
    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')
    
    model.load_state_dict(best_model_wts)
    return model, history

## 3. Phase A: Train Classifier Head Only

In [ ]:
model = get_model(num_classes=4, pretrained=True)
model = freeze_base_model(model)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
# Only optimize the classifier parameters
optimizer_head = optim.Adam(model.classifier.parameters(), lr=0.001)

dataloaders = {'train': train_loader, 'val': val_loader}

print("Starting Phase A: Training classifier head...")
model, history_a = train_model(model, dataloaders, criterion, optimizer_head, num_epochs=5, device=device)

## 4. Phase B: Unfreeze Last 3 Blocks and Fine-tune

In [ ]:
model = unfreeze_last_blocks(model, num_blocks_to_unfreeze=3)

# Now optimize all parameters that require gradients
params_to_update = [p for p in model.parameters() if p.requires_grad]
# Use a smaller learning rate for fine-tuning
optimizer_fine = optim.Adam(params_to_update, lr=1e-4)

print("Starting Phase B: Fine-tuning last 3 blocks...")
model, history_b = train_model(model, dataloaders, criterion, optimizer_fine, num_epochs=10, device=device)

## 5. Evaluate on Test Set & Save Weights

In [ ]:
# Combine histories
history = {
    'train_loss': history_a['train_loss'] + history_b['train_loss'],
    'val_loss': history_a['val_loss'] + history_b['val_loss'],
    'train_acc': history_a['train_acc'] + history_b['train_acc'],
    'val_acc': history_a['val_acc'] + history_b['val_acc'],
}

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Val Accuracy')
plt.axvline(x=4, color='r', linestyle='--', label='Unfreeze Point')
plt.legend()
plt.title('Accuracy over Epochs')

plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.axvline(x=4, color='r', linestyle='--', label='Unfreeze Point')
plt.legend()
plt.title('Loss over Epochs')
plt.show()

In [ ]:
# Test Set Evaluation
model.eval()
running_corrects = 0

for inputs, labels in test_loader:
    inputs = inputs.to(device)
    labels = labels.to(device)
    
    with torch.no_grad():
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        running_corrects += torch.sum(preds == labels.data)

test_acc = running_corrects.double() / len(test_loader.dataset)
print(f'Test Accuracy: {test_acc:.4f}')

# Save Model Weights
save_path = 'backend/models/efficientnet_adni.pth'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(model.state_dict(), save_path)
print(f"Model weights saved to {save_path}")